# 01 — Product Image Classifier Training
Module A1 (OpenCV basics) + A2 (transfer-learning classifier with MobileNetV2).
Run this from the `notebooks/` folder — it saves artifacts into `../app/models/`.


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

MODEL_DIR = "../app/models"
os.makedirs(MODEL_DIR, exist_ok=True)


## A1. OpenCV basics — quick sanity check of the preprocessing utilities

In [ ]:
from app.services.cv_utils import to_grayscale, resize_image, apply_blur, canny_edges, detect_faces

# Any local image works here — replace with a real product photo if you have one
sample_path = "../data/sample.jpg"
if not os.path.exists(sample_path):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg", sample_path
    )

img = cv2.imread(sample_path)
gray, blur, edges = to_grayscale(img), apply_blur(img, 7), canny_edges(img)
boxed, faces = detect_faces(img)

fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax, im, t in zip(
    axs,
    [gray, cv2.cvtColor(blur, cv2.COLOR_BGR2RGB), edges, cv2.cvtColor(boxed, cv2.COLOR_BGR2RGB)],
    ["Grayscale", "Blurred", "Canny Edges", f"Faces: {len(faces)}"],
):
    ax.imshow(im, cmap="gray" if im.ndim == 2 else None); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()


## A2. Product image classifier
Uses CIFAR-10 as a stand-in 5+-class retail dataset. **Swap the data loader below** for your own
Kaggle "Retail Product Checkout Dataset" or a folder of labeled product photos when you have one —
everything downstream (training loop, saving, the FastAPI route) stays the same.


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

N_TRAIN, N_TEST = 6000, 1000
x_train, y_train = x_train[:N_TRAIN], y_train[:N_TRAIN]
x_test, y_test = x_test[:N_TEST], y_test[:N_TEST]

IMG_SIZE = 96

def preprocess(x, y):
    x = tf.image.resize(x, (IMG_SIZE, IMG_SIZE))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    return x, y

train_ds = (tf.data.Dataset.from_tensor_slices((x_train, y_train))
            .map(preprocess).shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE))
test_ds = (tf.data.Dataset.from_tensor_slices((x_test, y_test))
           .map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE))

base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False

product_classifier = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(len(class_names), activation="softmax"),
])
product_classifier.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
product_classifier.summary()


In [ ]:
history = product_classifier.fit(train_ds, validation_data=test_ds, epochs=3)
loss, acc = product_classifier.evaluate(test_ds)
print(f"Test accuracy: {acc:.2%}")

product_classifier.save(f"{MODEL_DIR}/product_classifier.h5")
with open(f"{MODEL_DIR}/class_names.json", "w") as f:
    json.dump(class_names, f)
print("Saved product_classifier.h5 + class_names.json to", MODEL_DIR)
